# Experimento 003 - Pipeline TAO via API

Este notebook executa o fluxo completo usando `requests.Session` contra o NVIDIA TAO FTMS:

`train -> evaluate -> export -> gen_trt_engine -> inference`

Cada etapa fica em uma celula propria. A proxima etapa somente deve ser executada depois que a anterior terminar em `Done` ou `Completed`. Credenciais sao lidas automaticamente do `secrets.json` privado e nunca sao salvas no notebook.

## 1. Configuracao

Define endpoint, organizacao, workspace, dataset, modelo base e pasta de evidencias. Esta celula nao cria job.

In [27]:
import json
import os
import requests
import time
from datetime import datetime, timezone
from pathlib import Path

TAO_BASE_URL = os.getenv("TAO_BASE_URL", "http://100.107.81.126:8090").rstrip("/")
TAO_ORG = os.getenv("TAO_ORG", "getter")
WORKSPACE_ID = os.getenv("TAO_TEST_WORKSPACE_ID", "98c4bec1-88df-423e-8a15-af78881e9742")
DATASET_URI = os.getenv("TAO_TRAIN_DATASET_URI", "seaweedfs://tao-storage/data/tao-test-classification-v1")
BASE_EXPERIMENT_ID = os.getenv("TAO_BASE_EXPERIMENT_ID", "469f42fe-37dc-55ff-b89b-5096477f9a87")
EVIDENCE_DIR = Path(os.getenv("TAO_EXPERIMENT_003_EVIDENCE", "experiment-003-evidence"))
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
configured_secrets = os.getenv("TAO_SECRETS_FILE", "")
secret_candidates = [Path(configured_secrets)] if configured_secrets else []
secret_candidates += [Path("secrets.json"), Path.cwd() / "secrets.json", Path.cwd().parent / "secrets.json", Path("/home/getter/tao-ftms/secrets.json")]
secrets_file = next((candidate for candidate in secret_candidates if candidate.exists()), None)
if secrets_file is not None:
    NGC_KEY = json.loads(secrets_file.read_text(encoding="utf-8"))["ngc_api_key"]
else:
    NGC_KEY = os.getenv("NGC_KEY", "")
if not NGC_KEY:
    raise RuntimeError("NGC key not found in secrets.json or NGC_KEY")
print({"base_url": TAO_BASE_URL, "org": TAO_ORG, "workspace": WORKSPACE_ID, "dataset": DATASET_URI, "evidence": str(EVIDENCE_DIR)})

{'base_url': 'http://100.107.81.126:8090', 'org': 'getter', 'workspace': '98c4bec1-88df-423e-8a15-af78881e9742', 'dataset': 'seaweedfs://tao-storage/data/tao-test-classification-v1', 'evidence': 'experiment-003-evidence'}


## 2. Login e health

O login usa a Personal API Key e retorna um JWT mantido somente em memoria. O health confirma liveness e readiness do FTMS.

In [28]:
http = requests.Session()

def requests_json(method, path, payload=None, headers=None):
    request_headers = {"Content-Type": "application/json"}
    if headers:
        request_headers.update(headers)
    response = http.request(method, f"{TAO_BASE_URL}{path}", headers=request_headers, json=payload, timeout=(10, 120))
    response.raise_for_status()
    return response.json()

login = requests_json("POST", "/api/v2/login", {"ngc_key": NGC_KEY, "ngc_org_name": TAO_ORG, "enable_telemetry": False}, {"ngc_key": NGC_KEY})
TOKEN = login["token"]
AUTH = {"Authorization": f"Bearer {TOKEN}"}
health = requests_json("GET", "/api/v2/health")
assert "readiness" in health
print("login=OK health=OK")

login=OK health=OK


## 3. Contexto, workspace, dataset e schema

Antes de criar qualquer job, validamos os recursos e o contrato da action.

In [29]:
workspace = requests_json("GET", f"/api/v2/orgs/{TAO_ORG}/workspaces/{WORKSPACE_ID}", headers=AUTH)
datasets = requests_json("GET", f"/api/v2/orgs/{TAO_ORG}/datasets", headers=AUTH)
schema = requests_json("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:schema?network_arch=classification_pyt&action=train", headers=AUTH)
base_experiments = requests_json("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:list_base_experiments?network_arch=classification_pyt", headers=AUTH)
assert workspace.get("id") == WORKSPACE_ID
assert schema.get("type") == "object"
assert any(item.get("id") == BASE_EXPERIMENT_ID for item in base_experiments.get("experiments", []))
print("workspace=OK datasets=%d schema=OK base_experiment=OK" % len(datasets.get("datasets", [])))

workspace=OK datasets=1 schema=OK base_experiment=OK


## 4. Funcoes de payload e polling

As funcoes abaixo montam o payload de cada action e aguardam estado terminal. O polling consulta a API a cada 10 segundos e salva o ultimo JSON de status.

In [30]:
terminal_success = {"Done", "Completed"}
terminal_failure = {"Error", "Failed", "Failure", "Canceled", "Cancelled"}

def load_specs(action):
    env_name = {"train": "TAO_TRAIN_SPECS_JSON_FILE", "evaluate": "TAO_EVAL_SPECS_JSON_FILE", "export": "TAO_EXPORT_SPECS_JSON_FILE", "gen_trt_engine": "TAO_TRT_SPECS_JSON_FILE", "inference": "TAO_INFERENCE_SPECS_JSON_FILE"}[action]
    candidates = []
    if os.getenv(env_name): candidates.append(Path(os.environ[env_name]))
    candidates.append(Path(f"{action}_specs.json"))
    candidates.append(Path(f"/home/getter/tao-real-tests-20261001/{action}_specs.json"))
    for candidate in candidates:
        if candidate.exists(): return json.loads(candidate.read_text(encoding="utf-8"))
    return schema.get("default", {})

def make_payload(action, specs, parent_id=None):
    payload = {"kind": "experiment", "name": f"curl-experiment-003-{action}-{int(time.time())}", "network_arch": "classification_pyt", "encryption_key": "unittest-only", "workspace": WORKSPACE_ID, "action": action, "specs": specs}
    if parent_id: payload["parent_job_id"] = parent_id
    if action == "train":
        payload.update(train_dataset_uris=[DATASET_URI], eval_dataset_uri=DATASET_URI, inference_dataset_uri=DATASET_URI, base_experiment_ids=[BASE_EXPERIMENT_ID])
    elif action in {"evaluate", "export"}: payload["eval_dataset_uri"] = DATASET_URI
    elif action == "inference": payload["inference_dataset_uri"] = DATASET_URI
    return payload

def inspect_job(action, job_id, current):
    details = next(iter(current.get("job_details", {}).values()), {})
    try:
        files = requests_json("GET", f"/api/v2/orgs/{TAO_ORG}/jobs/{job_id}:list_files", headers=AUTH)
    except Exception as error:
        files = {"available": False, "error": str(error)}
    (EVIDENCE_DIR / f"{action}-files.json").write_text(json.dumps(files, indent=2), encoding="utf-8")
    evidence = {"action": action, "job_id": job_id, "status": current.get("status"), "created_on": current.get("created_on"), "last_modified": current.get("last_modified"), "parent_id": current.get("parent_id"), "dataset_uris": {"train": current.get("train_dataset_uris"), "eval": current.get("eval_dataset_uri"), "inference": current.get("inference_dataset_uri")}, "job_details": details, "artifacts": files}
    (EVIDENCE_DIR / f"{action}-evidence.json").write_text(json.dumps(evidence, indent=2), encoding="utf-8")
    print(json.dumps({"action": action, "job_id": job_id, "status": current.get("status"), "kpi": details.get("kpi", []), "detailed_status": details.get("detailed_status"), "artifact_listing": files}, indent=2))

def submit_and_wait(action, parent_id=None):
    specs = load_specs(action)
    if action in {"export", "gen_trt_engine", "inference"}: specs.setdefault("dataset", {})["classes_file"] = DATASET_URI.rstrip("/") + "/classes.txt"
    if action in {"gen_trt_engine", "inference"}: specs.setdefault("dataset", {})["batch_size"] = 1
    payload = make_payload(action, specs, parent_id)
    (EVIDENCE_DIR / f"{action}-payload.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
    submitted = requests_json("POST", f"/api/v2/orgs/{TAO_ORG}/jobs", payload, AUTH)
    job_id = submitted["id"]
    print(action, "submitted", job_id)
    deadline = time.monotonic() + int(os.getenv("TAO_JOB_TIMEOUT_SECONDS", "3600"))
    while time.monotonic() < deadline:
        current = requests_json("GET", f"/api/v2/orgs/{TAO_ORG}/jobs/{job_id}", headers=AUTH)
        (EVIDENCE_DIR / f"{action}-status.json").write_text(json.dumps(current, indent=2), encoding="utf-8")
        state = current.get("status")
        print(action, state)
        if state in terminal_success:
            logs_response = http.get(f"{TAO_BASE_URL}/api/v2/orgs/{TAO_ORG}/jobs/{job_id}:logs", headers=AUTH, timeout=(10, 120))
            if logs_response.ok:
                (EVIDENCE_DIR / f"{action}-logs.txt").write_text(logs_response.text, encoding="utf-8")
            inspect_job(action, job_id, current)
            return job_id, current
        if state in terminal_failure:
            raise RuntimeError(f"{action} failed: {job_id}")
        time.sleep(10)
    raise TimeoutError(f"{action} timeout: {job_id}")

## 5. Train

Treina o modelo `classification_pyt` por uma epoca em uma GPU. O checkpoint gerado sera usado pelas actions seguintes.

In [31]:
train_job_id, train_status = submit_and_wait("train")
train_metrics = next(iter(train_status.get("job_details", {}).values()), {}).get("kpi", [])
print("train Done", train_job_id, train_metrics)

train submitted c45a3fe8-858d-4e18-a7b5-385cc474aa90
train Pending
train Pending
train Started
train Started
train Started
train Started
train Started
train Started
train Started
train Running
train Running
train Running
train Done
{
  "action": "train",
  "job_id": "c45a3fe8-858d-4e18-a7b5-385cc474aa90",
  "status": "Done",
  "kpi": [
    {
      "metric": "val_loss",
      "values": {
        "0": 2.090709686279297,
        "1": 1.6890172958374023,
        "2": 1.3091542720794678,
        "3": 1.0176483392715454,
        "4": 0.8069472908973694,
        "5": 0.6671432256698608,
        "6": 0.578280508518219,
        "7": 0.5187190771102905,
        "8": 0.4794667065143585,
        "9": 0.4794667065143585
      }
    },
    {
      "metric": "val_acc_1",
      "values": {
        "0": 1.0,
        "1": 1.0,
        "2": 1.0,
        "3": 1.0,
        "4": 1.0,
        "5": 1.0,
        "6": 1.0,
        "7": 1.0,
        "8": 1.0,
        "9": 1.0
      }
    },
    {
      "metric":

## 6. Evaluate

Avalia o checkpoint produzido pelo treino no dataset de avaliacao configurado.

In [32]:
evaluate_job_id, evaluate_status = submit_and_wait("evaluate", train_job_id)
print("evaluate Done", evaluate_job_id, next(iter(evaluate_status.get("job_details", {}).values()), {}).get("kpi", []))

evaluate submitted fe1af6bc-6d28-4da2-809b-9337ef3b6603
evaluate Pending
evaluate Started
evaluate Started
evaluate Started
evaluate Started
evaluate Started
evaluate Started
evaluate Started
evaluate Done
{
  "action": "evaluate",
  "job_id": "fe1af6bc-6d28-4da2-809b-9337ef3b6603",
  "status": "Done",
  "kpi": [
    {
      "metric": "val_acc_1",
      "values": {
        "0": 1.0
      }
    }
  ],
  "detailed_status": {
    "date": "10/19/2026",
    "time": "19:24:27",
    "message": "evaluate action completed successfully for classification_pyt",
    "status": "SUCCESS"
  },
  "artifact_listing": [
    "results/fe1af6bc-6d28-4da2-809b-9337ef3b6603/evaluate/experiment.yaml",
    "results/fe1af6bc-6d28-4da2-809b-9337ef3b6603/evaluate/lightning_logs/version_0/events.out.tfevents.1792437863.b0ca2c4d77f0.286.0",
    "results/fe1af6bc-6d28-4da2-809b-9337ef3b6603/evaluate/lightning_logs/version_0/hparams.yaml",
    "results/fe1af6bc-6d28-4da2-809b-9337ef3b6603/evaluate/result.csv",
    "r

## 7. Export ONNX

Converte o checkpoint treinado para ONNX. Esta etapa preserva o parent job de treino e exige os metadados do dataset, incluindo `classes.txt`.

In [33]:
export_job_id, export_status = submit_and_wait("export", train_job_id)
print("export Done", export_job_id)

export submitted 0339296f-f5f0-4618-a61b-f136eb09a5fa
export Pending
export Pending
export Started
export Started
export Started
export Started
export Started
export Started
export Started
export Done
{
  "action": "export",
  "job_id": "0339296f-f5f0-4618-a61b-f136eb09a5fa",
  "status": "Done",
  "kpi": [],
  "detailed_status": {
    "date": "10/19/2026",
    "time": "19:25:56",
    "message": "export action completed successfully for classification_pyt",
    "status": "SUCCESS"
  },
  "artifact_listing": [
    "results/0339296f-f5f0-4618-a61b-f136eb09a5fa/export/experiment.yaml",
    "results/0339296f-f5f0-4618-a61b-f136eb09a5fa/export/status.json",
    "results/0339296f-f5f0-4618-a61b-f136eb09a5fa/microservices_log.txt",
    "results/0339296f-f5f0-4618-a61b-f136eb09a5fa/model.onnx"
  ]
}
export Done 0339296f-f5f0-4618-a61b-f136eb09a5fa


## 8. Gerar TensorRT

Gera o engine TensorRT a partir do modelo exportado. O engine sera o parent da inferencia. A celula tambem consulta `:list_files`; os artefatos ficam no armazenamento configurado do FTMS/SeaweedFS e a lista retornada pelo TAO e salva em `experiment-003-evidence/gen_trt_engine-files.json`.

In [34]:
trt_job_id, trt_status = submit_and_wait("gen_trt_engine", export_job_id)
print("gen_trt_engine Done", trt_job_id)

gen_trt_engine submitted a24e9173-5d63-45e8-8a85-0639b47a7648
gen_trt_engine Pending
gen_trt_engine Started
gen_trt_engine Started
gen_trt_engine Started
gen_trt_engine Started
gen_trt_engine Started
gen_trt_engine Started
gen_trt_engine Started
gen_trt_engine Running
gen_trt_engine Running
gen_trt_engine Running
gen_trt_engine Done
{
  "action": "gen_trt_engine",
  "job_id": "a24e9173-5d63-45e8-8a85-0639b47a7648",
  "status": "Done",
  "kpi": [],
  "detailed_status": {
    "date": "10/19/2026",
    "time": "19:28:2",
    "message": "gen_trt_engine action completed successfully for classification_pyt",
    "status": "SUCCESS"
  },
  "artifact_listing": [
    "results/a24e9173-5d63-45e8-8a85-0639b47a7648/gen_trt_engine/experiment.yaml",
    "results/a24e9173-5d63-45e8-8a85-0639b47a7648/gen_trt_engine/status.json",
    "results/a24e9173-5d63-45e8-8a85-0639b47a7648/microservices_log.txt",
    "results/a24e9173-5d63-45e8-8a85-0639b47a7648/model.engine"
  ]
}
gen_trt_engine Done a24e9173-5d

## 9. Inference

Executa inferencia TensorRT no dataset de inferencia, usando o engine produzido na etapa anterior. O resultado, score e predictions somente serao exibidos se o TAO os fornecer em `job_details`, logs ou arquivos. Ausencia de score significa `nao fornecido pelo TAO`, nunca zero. A lista de saida fica em `experiment-003-evidence/inference-files.json`.

In [35]:
inference_job_id, inference_status = submit_and_wait("inference", trt_job_id)
print("inference Done", inference_job_id)

inference submitted b6ef559a-38a5-4d6f-a293-cd64c88b3c82
inference Pending
inference Started
inference Started
inference Started
inference Started
inference Started
inference Started
inference Started
inference Done
{
  "action": "inference",
  "job_id": "b6ef559a-38a5-4d6f-a293-cd64c88b3c82",
  "status": "Done",
  "kpi": [],
  "detailed_status": {
    "date": "10/19/2026",
    "time": "19:29:16",
    "message": "inference action completed successfully for classification_pyt",
    "status": "SUCCESS"
  },
  "artifact_listing": [
    "results/b6ef559a-38a5-4d6f-a293-cd64c88b3c82/inference_results.tar.gz",
    "results/b6ef559a-38a5-4d6f-a293-cd64c88b3c82/microservices_log.txt",
    "results/b6ef559a-38a5-4d6f-a293-cd64c88b3c82/trt_inference/status.json"
  ]
}
inference Done b6ef559a-38a5-4d6f-a293-cd64c88b3c82


## 10. Definition of Done

O experimento esta concluido somente quando as cinco actions terminarem com sucesso e os JSONs de status estiverem salvos em `experiment-003-evidence/`.

In [36]:
completed = {"train": train_job_id, "evaluate": evaluate_job_id, "export": export_job_id, "gen_trt_engine": trt_job_id, "inference": inference_job_id}
artifact_summary = {}
for action in completed:
    evidence_file = EVIDENCE_DIR / f"{action}-evidence.json"
    evidence = json.loads(evidence_file.read_text(encoding="utf-8")) if evidence_file.exists() else {}
    artifact_summary[action] = {"job_id": completed[action], "kpi": evidence.get("job_details", {}).get("kpi", []), "artifact_listing": evidence.get("artifacts", {}), "api_download_endpoint": f"{TAO_BASE_URL}/api/v2/orgs/{TAO_ORG}/jobs/{completed[action]}:download"}
summary = {"experiment": "003", "status": "PASSED", "jobs": completed, "artifacts_and_metrics": artifact_summary, "finished_at": datetime.now(timezone.utc).isoformat()}
(EVIDENCE_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

{
  "experiment": "003",
  "status": "PASSED",
  "jobs": {
    "train": "c45a3fe8-858d-4e18-a7b5-385cc474aa90",
    "evaluate": "fe1af6bc-6d28-4da2-809b-9337ef3b6603",
    "export": "0339296f-f5f0-4618-a61b-f136eb09a5fa",
    "gen_trt_engine": "a24e9173-5d63-45e8-8a85-0639b47a7648",
    "inference": "b6ef559a-38a5-4d6f-a293-cd64c88b3c82"
  },
  "artifacts_and_metrics": {
    "train": {
      "job_id": "c45a3fe8-858d-4e18-a7b5-385cc474aa90",
      "kpi": [
        {
          "metric": "val_loss",
          "values": {
            "0": 2.090709686279297,
            "1": 1.6890172958374023,
            "2": 1.3091542720794678,
            "3": 1.0176483392715454,
            "4": 0.8069472908973694,
            "5": 0.6671432256698608,
            "6": 0.578280508518219,
            "7": 0.5187190771102905,
            "8": 0.4794667065143585,
            "9": 0.4794667065143585
          }
        },
        {
          "metric": "val_acc_1",
          "values": {
            "0": 1.0,